In [0]:
%pip install --quiet mlflow==2.19
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("db", "", "Database")
dbutils.widgets.text("model_name", "", "Model")
dbutils.widgets.text("validation_dataset", "", "validation dataset")

In [0]:
catalog = dbutils.widgets.get("catalog")
db = dbutils.widgets.get("db")
model_name = dbutils.widgets.get("model_name")
validation_dataset = dbutils.widgets.get("validation_dataset")

In [0]:
# We are interested in validating the Challenger model
import mlflow
from mlflow.tracking import MlflowClient
model_alias = "Challenger"
full_model_name = f"{catalog}.{db}.{model_name}"

client = MlflowClient()
model_details = client.get_model_version_by_alias(full_model_name, model_alias)
model_version = int(model_details.version)

print(f"Validating {model_alias} model for {full_model_name} on model version {model_version}")

In [0]:
# If there's no description or an insufficient number of charaters, tag accordingly
if not model_details.description:
  has_description = False
  print("Please add model description")
elif not len(model_details.description) > 20:
  has_description = False
  print("Please add detailed model description (40 char min).")
else:
  has_description = True

print(f'Model {full_model_name} version {model_details.version} has description: {has_description}')
client.set_model_version_tag(name=full_model_name, version=str(model_details.version), key="has_description", value=has_description)

In [0]:
model_run_id = model_details.run_id
test_smape = mlflow.get_run(model_run_id).data.metrics['test_smape']

try:
    #Compare the challenger smape score to the existing champion if it exists
    champion_model = client.get_model_version_by_alias(full_model_name, "Champion")
    champion_smape = mlflow.get_run(champion_model.run_id).data.metrics['test_smape']
    print(f'Champion test smape score: {champion_smape}. Challenger test smape score: {test_smape}.')
    metric_smape_passed = test_smape >= champion_smape
except:
    print(f"No Champion found. Accept the model as it's the first one.")
    metric_smape_passed = True

print(f'Model {full_model_name} version {model_details.version} metric_smape_passed: {metric_smape_passed}')
# Tag that F1 metric check has passed
client.set_model_version_tag(name=full_model_name, version=model_details.version, key="metric_smape_passed", value=metric_smape_passed)

In [0]:
import pyspark.sql.functions as F
#get our validation dataset:
validation_df = spark.table(f"{validation_dataset}")

#Call the model with the given alias and return the prediction
def predict_model(validation_df, model_alias):
    model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@{model_alias}")
    return validation_df.withColumn('prediction', model(*model.metadata.get_input_schema().input_names()))

In [0]:
!pip install mlflow==2.9.2 gluonts[torch]==0.15.1 databricks-automl-runtime==0.2.20.6

In [0]:
%pip install --force-reinstall databricks-automl-runtime==0.2.20.6 mlflow==2.9.2 gluonts[torch]==0.15.1

In [0]:
import mlflow
from pyspark.sql.functions import struct, col
import pyspark.sql.functions as F
import pandas as pd
#runs:/2eac328bda5e4f078c9a2a9dc32eba6b/model
#model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@{model_alias}")
model = mlflow.pyfunc.load_model(model_uri=f"models:/{catalog}.{db}.{model_name}@{model_alias}")


In [0]:
a = model.predict(validation_df)

In [0]:
a_df = spark.createDataFrame(a)
a_df = a_df.withColumnRenamed("yhat", "prediction")

In [0]:
a_df.write.saveAsTable(f"{catalog}.{db}.validation_table")

In [0]:
%sql
select * from hg.forecasting_interest.validation_table

In [0]:
_sqldf = _sqldf.withColumnRenamed("yhat", "actual")

In [0]:
joined_df = a_df.join(_sqldf, a_df["Time_Period"] == a_df["Time_Period"], "inner")
display(joined_df)

In [0]:
joined_df = joined_df.drop("Time_Period")
display(joined_df)

In [0]:
validation_df = spark.table(f"{validation_dataset}").toPandas()

In [0]:
predictions_df = predict_model(validation_df, "challenger")

In [0]:
df.withColumn('predictions', loaded_model(struct(*map(col, df.columns))))

In [0]:
import pandas as pd
import numpy as np

def smape(df, actual_col, predicted_col):
  """
  Calculates the Symmetric Mean Absolute Percentage Error (SMAPE).

  Args:
    df: Pandas DataFrame containing actual and predicted values.
    actual_col: Name of the column containing actual values.
    predicted_col: Name of the column containing predicted values.

  Returns:
    The SMAPE value as a float.
  """
  actual = df[actual_col]
  predicted = df[predicted_col]
  return np.mean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted))) * 100

df = joined_df.toPandas()

val_smape_value = 1-smape(df, 'actual', 'prediction')
print(f"SMAPE: {val_smape_value:.2f}")

In [0]:
if val_smape_value < 0.05:

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
def calculate_smape(actual, prediction):
    return 100/len(actual) * sum(2 * abs(prediction - actual) / (abs(actual) + abs(prediction)))

# Assuming 'actual' and 'predictions' columns exist in predictions_df
smape_udf = F.udf(calculate_smape, FloatType())
smape_value = joined_df.select(smape_udf(F.col('actual'), F.col('prediction')).alias('smape')).collect()[0]['smape']

print(f'SMAPE value: {smape_value}')